## The power of the graph perspective in clustering

#### Dependencies

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

from scipy.linalg import svd
from scipy.spatial import distance
from sklearn.preprocessing import normalize

#### Helper Functions

In [ ]:
def get_data(n_samples, seed=170):
    T_matrix = [[-0.60834549, -0.63667341], [0.40887718, 0.85253229]]
    X_orig, y_orig = make_blobs(n_samples=n_samples, random_state=seed)
    X = np.dot(X_orig, T_matrix)
    return X

In [ ]:
def show_data_results(X, y_pred=None, cmap='jet'):
    if y_pred is None:
        plt.scatter(X[:, 0], X[:, 1])
        plt.title ("input data")
    else:
        fig = plt.figure(figsize=(10, 3))
        ax1 = fig.add_subplot(121)
        ax1.scatter(X[:, 0], X[:, 1])
        ax1.set(xticks=[],yticks=[],title ="input data")

        ax2 = fig.add_subplot(122)
        ax2.scatter(X[:, 0], X[:, 1], c=y_pred, cmap=cmap)
        ax2.set(xticks=[], yticks=[], title ="clustered data")

# K-Means Clustering

Please, read https://en.wikipedia.org/wiki/K-means_clustering about the Kmeans algorithm.

In this problem, we will show how interpreting a dataset as a graph may result in obtaining an elegant clustering solution. We have an input dataset that we wish to cluster in 3 apparent classes.

We provide the synthetic dataset of 2000 points described below where the T_matrix is just a 2D transformation matrix.

**Note:** please keep the provided `seed = 170` (and $\gamma = 7.5$ below). The written questions refer to the results at this setting; with other seeds the three blobs land in different places and the comparison between the methods below can come out differently.

In [ ]:
n_samples, seed = 2000, 170
X = get_data(n_samples, seed)

In [ ]:
show_data_results(X)

Using the the Kmeans algorithm implementation of sklearn, show your attempt to cluster this dataset into 3 classes in one luine of code.

In [ ]:
y_pred = KMeans(n_clusters=3, random_state=seed).fit_predict(X)

In [ ]:
show_data_results(X, y_pred)

## Q.1.

Comment on the output the the KMeans algorithm? Did it work? If so explain why, if not, explain not not.

## Q.2.

Let's now interpret every single point in the provided dataset as a node in a graph. Our goal is to find a way to relate every node in the graph in such way that the points that are closer together maintain that relationship while points that are far are explicitly identified. 

When lots of points are close on top of each other, kmeans is missing it. Representing this as a graph unveils the relation.

One way to capture such relationship between points (nodes) in a graph is through the Adjacency matrix. Typically, a simple adjacency matrix between nodes of an undirected graph is given by:

$$A_{i, j}=\begin{equation}
\left\{ 
  \begin{aligned}
    &1 \text{  if there is an edge between node i and node j}, \\
    &0 \text{  otherwise.}\\
  \end{aligned}
  \right.
\end{equation}
$$

In this problem, we will use the weighted distances between points instead as a similarity measure. Write a function that takes in the input dataset and some coefficient gamma which returns the adjacency matrix A.

$$
A_{i, j} = e^{-\gamma ||x_{i}-x_{j}||^2}
$$ 

where $x_{i}$ and $x_{j}$ represent each point in the provided dataset. You may find the `distance` module from `scipy.spatial` useful.

In [ ]:
def get_adjacency_matrix(gamma, X):
    # TODO: fill in your code here
    # adjacency_matrix = ?
    return adjacency_matrix

## Q. 3.

The degree matrix of an undirected graph is a diagonal matrix which contains information about the degree of each vertex. In other words, it contains the number of edges attached to each vertex and it is given by:

$$D_{i, j}=\begin{equation}
\left\{ 
  \begin{aligned}
    deg(v_{i}): \text{if i == j}, \\
    0: \text{otherwise.}\\
  \end{aligned}
  \right.
\end{equation}
$$

where the degree $ {\deg(v_{i})}$ of a vertex counts the number of times an edge terminates at that vertex. Note that in the traditional definition of the adjacency matrix, this boils down to the diagonal matrix in which element along the diagonals are column-wise sum of the adjacency matrix. Using the same idea, write a function that takes in the adjacency matrix as argument and returns the degrees, i.e. the diagonal of the degree matrix $D$, as a 1-D array of length $n$ (`degree_matrix[i]` $= \deg(v_i)$). You do not need to build the full $n \times n$ diagonal matrix; you will use this vector to form $D^{-1/2}$ in Q4.


In [ ]:
def get_degree_matrix(adjacency_matrix):
    # TODO: fill in your code here
    # degree_matrix = ?  (1-D array of degrees, shape (n,))
    return degree_matrix

## Q. 4.

Using $\gamma$ = 7.5, compute the adjacency matrix $A$, the degrees (diagonal of $D$), and the symmetrically normalized adjacency matrix $M = D^{-1/2} A D^{-1/2}$.

In [ ]:
# adjacency_matrix = ?
# degree_matrix = ?
# M = ?

## Q. 5.

Using SVD decomposition, write a function `cluster_data(M, num_clusters=3)` that selects the first 3 vectors (corresponding to the largest singular values) of the matrix U and performs the same KMeans clustering used above on them. It should return `(Usub, y_pred)`: the selected columns of U and the predicted cluster labels. Run it on $M$, look at the plot and at the printed cluster sizes. What do you observe? Did it work? If so explain why, if not, explain why not.

In [ ]:
def cluster_data(M, num_clusters=3):
    # TODO: fill in your code here
    # Compute the SVD of M (e.g. with scipy's svd, full_matrices=False), keep the first
    # num_clusters columns of U, and cluster the rows of Usub with
    # KMeans(n_clusters=num_clusters, random_state=seed), as above.
    # Usub = ?     (shape (n, num_clusters))
    # y_pred = ?   (shape (n,))
    return Usub, y_pred

In [ ]:
y_pred_spectral = cluster_data(M)[1]
show_data_results(X, y_pred_spectral)
print('cluster sizes:', np.bincount(y_pred_spectral))

## Q.6. 

Now let's think of the matrix $M$ obtained above as the transition matrix of a Markov Chain. To do so, $M$ needs to be a proper stochastic matrix which means that the sum of the elements in each column must add up to 1. Write a function that takes in the matrix M and returns M_stoch, the stochastic version of M; compute the stochastic matrix.

In [ ]:
def stochastic_matrix_converter(M):
    # TODO: fill in your code here
    # M_stoch = ?
    return M_stoch

In [ ]:
M_stoch = stochastic_matrix_converter(M)

Perform the same KMeans clustering used above on the resulting top 3 U vectors (use your `cluster_data` function from Q5) and plot your answers.

In [ ]:
y_pred_spectral_stoch = cluster_data(M_stoch)[1]
show_data_results(X, y_pred_spectral_stoch)
print('cluster sizes:', np.bincount(y_pred_spectral_stoch))